<h2>Food Recipe Website Scrape 1</h2>

<h3>Contents</h3>
<ul>
    <li><a id='#intro'>Introduction</a></li>
    <li><a id='#testing'>Testing</a></li>
    <li><a id='#code'>Scrapcode</a></li>
</ul>


<h3>Introduction</h3>
<a id='intro'></a>

<p>To be added</p>

In [7]:
#importing necessary packages

from bs4 import BeautifulSoup
import lxml
import requests
import numpy as np
import pandas as pd
print("packages imported successfully")

packages imported successfully


In [8]:
#assigning site to a variable
website = "https://www.bbcgoodfood.com"
websiteSearchPattern = "https://www.bbcgoodfood.com/search?q="

### Testing Section
<a id='testing'></a>
<p>This section primarily looks at testing what can be scraped from the website. Different aspects of the site will be scraped to see how they are done before proceeding to write the whole code to scrape multiple items from the site.</p>

In [9]:
#use requests package to get html content of the site and parse it in beautifulsoup with lxml
site_request = requests.get(websiteSearchPattern+"chicken")
site_soup = BeautifulSoup(site_request.text,'lxml')

#writing contents to a file
with open('Site content.txt','w') as file:
    file.write(site_soup.prettify())
    file.close()

In [10]:
#searching content for search results
recipes = site_soup.find_all('article')

Getting information from the first result

In [11]:
firstRecipe = recipes[0]

#getting name of recipe
recipeName = firstRecipe.find('h2',class_='heading-4').text

#getting the ratings
recipeRating = firstRecipe.find(class_='sr-only').text.split()[4]
reciptRatingCount = firstRecipe.find(class_='rating__count-text body-copy-small').text.split()[0]

print(f"The name of the recipe is {recipeName} and it has been rated up to {reciptRatingCount} times to obtain a rating of {recipeRating} out of 5")

The name of the recipe is Chicken & chorizo jambalaya and it has been rated up to 2662 to obtain a rating of 4.8 out of 5


In [13]:
#get recipe link

recipeLink = firstRecipe.find(class_='card__section card__media').a['href']
print("Link for recipe is " + recipeLink)

recipeSite = requests.get(website+recipeLink)
recipeSoup = BeautifulSoup(recipeSite.text,'lxml')

Link for recipe is /recipes/chicken-chorizo-jambalaya


In [16]:
#information from recipe link

recipePrepTime = int(recipeSoup.find_all('time')[0].text.split()[0])
recipeCookTime = int(recipeSoup.find_all('time')[1].text.split()[0])

print("Prep time: "+ str(recipePrepTime))
print("Cooking time: " + str(recipeCookTime))
print("Total time: " + str(recipePrepTime+recipeCookTime))

Prep time: 10
Cooking time: 45
Total time: 55


In [19]:
#get and store recipe image
recipeImg = recipeSoup.find(class_='post recipe').find('img', class_='image__img')['src']
ImgContent = requests.get(recipeImg).content

with open(recipeName+".png",'wb') as file:
    file.write(ImgContent)
    file.close()

In [28]:
recipeSoup.find_all('td')

[<td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">kcal</td>,
 <td class="key-value-blocks__value">445</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">fat</td>,
 <td class="key-value-blocks__value">10<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">saturates</td>,
 <td class="key-value-blocks__value">3<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">carbs</td>,
 <td class="key-value-blocks__value">64<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">sugars</td>,
 <td class="key-value-blocks__value">7<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">fibre</td>,
 <td class="key-value-blocks__value">2<!-- -->g</td>,
 <td class="key-value-blocks__prefix"></td>,
 <td class="key-value-blocks__key">protein</td>,
 <td class="key-value-blocks__val

In [29]:
test = recipeSoup.find_all('td')
print("number of items: "+str(len(test)))
for row in test:
    if row.text == 'kcal':
        print(row.find_next('td').text)



number of items: 24
445


Different Search patterns

<li>By Rating: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=rating</li>
<li>By Relevance: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=relevant</li>
<li>By Fastest Cook time: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=quickest</li>
<li>By Latest added: https://www.bbcgoodfood.com/search?q=chicken&tab=recipe&sort=published</li>

### ScarpCode <a id='code'></a>

<p>The code that will be written will be making use of the search pattern by rating and will only scrape the first page as it contains 30 results</p>

In [ ]:
'''
For this code, the search pattern used will be By Rating and only the first page will be scraped which should be the most rated foods
on the platform. The code will start by requesting for a food to search for and the results will be stored in a dictionary which will later on
be converted to a datatable using pandas.
'''
website = "https://www.bbcgoodfood.com"
search = input("Enter a meal/dish you want to search for: ")
websiteSearchPattern = f'https://www.bbcgoodfood.com/search?q={search}&tab=recipe&sort=rating'

#defining the dictionary
ScrapeResults = {}

site_request = requests.get(search)
site_soup = BeautifulSoup(site_request.text,'lxml')


recipes_results = site_soup.find_all('article')

#looping through all the results
for recipe in recipes_results:

    ScrapeResults[recipe.index]["Name"] = recipe.find('h2',class_='heading-4').text
    ScrapeResults[recipe.index]["Rating"] = recipe.find(class_='sr-only').text.split()[4]
    ScrapeResults[recipe.index]["Review Count"] = recipe.find(class_='rating__count-text body-copy-small').text.split()[0]

    ScrapeResults[recipe.index]["link"] = website+recipe.find(class_='card__section card__media').a['href']
    print("Link for recipe is " + ScrapeResults[recipe.index]["link"])

    #dive into each recipe
    print('getting more information for recipe unique link')
    recipeSite = requests.get(ScrapeResults[recipe.index]["link"])
    recipeSoup = BeautifulSoup(recipeSite.text,'lxml')

    ScrapeResults[recipe.index]["Prep time"] = int(recipeSoup.find_all('time')[0].text.split()[0])
    ScrapeResults[recipe.index]["Cooking time"] = int(recipeSoup.find_all('time')[1].text.split()[0])
    ScrapeResults[recipe.index]["Total time"] = ScrapeResults[recipe.index]["Prep time"] + ScrapeResults[recipe.index]["Cooking time"]


    tablesearch = recipeSoup.find_all('td')
    print("number of items: "+str(len(tablesearch)))
    for row in tablesearch:
        if row.text == 'kcal':
            calories = row.find_next('td').text
            print("number of calories is " + calories + "kcal")
            ScrapeResults[recipe.index]["kcal"] = calories

In [ ]:
ScrapeResults

This next section will be making use of pandas library to convert the dictionary into a table and see if any analytics can be done